### Ситников Сергей Олегович М3239
# **Лабораторная 1**

## Задача 1 (Вариант 1)
В городе с населением $n+1$ человек некто узнает новость. Он передает ее первому встречному, тот еще одному и т.д. На каждом шаге услышавший новость может сообщить её любому из $n$ человек с одинаковыми вероятностями. Найти вероятность того, что в течение $r$ единиц времени новость никто не услышит дважды. Решить ту же задачу в предположении, что на каждом шаге услышавший на предыдущей итерации сообщение передает его группе из $m$ человек.

В первой версии на $j$-том шаге у нас $j$ человек осведомлены о новости, остаётся $n+1-j$ людей, которые неосведомлены. Чтобы на $r$-ом шаге никто не узнал новость дважды, эта новость должна быть произнесена за всё время только тем, кто о ней ещё не был осведомлен. Итого формула выглядит как 
$$ P(r) = \frac{n}{n} * \frac{n - 1}{n} * \frac{n - 2}{n} * \cdots * \frac{n - r + 1}{n} = \prod_{k=1}^{r} \frac{n-k+1}{n}=\frac{n!}{(n-r)!n^r},\quad 0 \le r \le n$$
При $r>n$ вероятность равна нулю.

In [21]:
import random
import math
random.seed("Sitnikov Sergey")

def simulate_one(n: int, r: int) -> bool:
    knows = [False] * (n + 1)
    knows[0] = True
    current_teller = 0
    for _ in range(r):
        index_chosen = random.randint(0, n - 1)
        if index_chosen >= current_teller:
            index_chosen += 1
        if knows[index_chosen]:
            return False
        knows[index_chosen] = True
        current_teller = index_chosen
    return True

COUNT_EXPERIMENTS = 200_000
print("n     r      simulation    formula       delta to formula")
for n in range(4, 10):
    for r in range(1, n + 1):
        good_count = 0
        for _ in range(COUNT_EXPERIMENTS):
            good_count += simulate_one(n, r)
        simulation = good_count / COUNT_EXPERIMENTS
        formula = math.factorial(n) / math.factorial(n - r) / (n**r)
        delta = (simulation - formula) / formula * 100
        print(f"{n:<3} {r:>3} {simulation:>13.6f} {formula:>13.6f} {delta:+10.2f}%")

n     r      simulation    formula       delta to formula
4     1      1.000000      1.000000      +0.00%
4     2      0.751525      0.750000      +0.20%
4     3      0.374110      0.375000      -0.24%
4     4      0.094495      0.093750      +0.79%
5     1      1.000000      1.000000      +0.00%
5     2      0.800570      0.800000      +0.07%
5     3      0.481250      0.480000      +0.26%
5     4      0.191185      0.192000      -0.42%
5     5      0.038530      0.038400      +0.34%
6     1      1.000000      1.000000      +0.00%
6     2      0.832680      0.833333      -0.08%
6     3      0.556110      0.555556      +0.10%
6     4      0.277125      0.277778      -0.24%
6     5      0.092230      0.092593      -0.39%
6     6      0.015110      0.015432      -2.09%
7     1      1.000000      1.000000      +0.00%
7     2      0.856665      0.857143      -0.06%
7     3      0.612035      0.612245      -0.03%
7     4      0.351735      0.349854      +0.54%
7     5      0.149465      0.1

Теперь вторая версия задачи, теперь сообщается новость не одному человеку за раз, а сразу m людям. Тогда на $j$-ом шагу нужно выбрать m людей только среди тех, кто еще не осведомлён. Неосведомлённых людей $n - (j-1)m$ (на каждом шагу осведомляется на m людей больше, это всё при учёте, что мы не хотим, чтобы одному и тому же человеку дважды была рассказана новость). Передатчик на $j$-ом шаге выбирается среди тех, кому рассказали новость на $(j-1)$-ом шаге. Значит формула будет выглядеть так:
$$ P(r) = \prod_{j=1}^r \frac{C_{n-(j-1)m}^m}{C_n^m}, \quad mr \le n$$
При $mr > n$ вероятность равна нулю.

In [22]:
import random
import math
random.seed("Sitnikov Sergey")

def simulate_one(n: int, m: int, r: int) -> bool:
    knows = [False] * (n + 1)
    knows[0] = True
    current_teller = 0
    for _ in range(r):
        possible_receivers = [
            person for person in range(n + 1)
            if person != current_teller
        ]
        group = random.sample(possible_receivers, m)
        for person in group:
            if knows[person]:
                return False
            knows[person] = True
        current_teller = random.choice(group)
    return True


COUNT_EXPERIMENTS = 200_000
print("n     m   r      simulation    formula       delta to formula")

for n in range(4, 10):
    for m in range(1, n + 1):
        for r in range(1, n // m + 1):
            good_count = 0
            for _ in range(COUNT_EXPERIMENTS):
                good_count += simulate_one(n, m, r)
            simulation = good_count / COUNT_EXPERIMENTS
            formula = math.prod(
                math.comb(n - (j - 1) * m, m) / math.comb(n, m)
                for j in range(1, r + 1)
            )
            delta = (simulation - formula) / formula * 100
            print(
                f"{n:<3} {m:>3} {r:>3} "
                f"{simulation:>13.6f} {formula:>13.6f} "
                f"{delta:+10.2f}%"
            )

n     m   r      simulation    formula       delta to formula
4     1   1      1.000000      1.000000      +0.00%
4     1   2      0.750490      0.750000      +0.07%
4     1   3      0.375810      0.375000      +0.22%
4     1   4      0.093080      0.093750      -0.71%
4     2   1      1.000000      1.000000      +0.00%
4     2   2      0.166465      0.166667      -0.12%
4     3   1      1.000000      1.000000      +0.00%
4     4   1      1.000000      1.000000      +0.00%
5     1   1      1.000000      1.000000      +0.00%
5     1   2      0.799920      0.800000      -0.01%
5     1   3      0.479905      0.480000      -0.02%
5     1   4      0.191495      0.192000      -0.26%
5     1   5      0.038300      0.038400      -0.26%
5     2   1      1.000000      1.000000      +0.00%
5     2   2      0.297990      0.300000      -0.67%
5     3   1      1.000000      1.000000      +0.00%
5     4   1      1.000000      1.000000      +0.00%
5     5   1      1.000000      1.000000      +0.00%
6 

## Задача 2 (Вариант 2)
Случайная точка $A$ имеет равномерное распределение в правильном
$n$-угольнике. Найти вероятность $P_n$, что точка $A$ находится
ближе к границе многоугольника, чем к его диагоналям.
Найти числа $C$, $a$, что
$$P_n = Cn^a(1 + o(1)), \qquad n \to \infty.$$

![second](img/second.png)

Рассмотрим правильный $n$-угольник. Пусть длина стороны будет $s$. Занумеруем вершины: $v_0$, $v_1$, $\cdots$, $v_{n-1}$. Рассмотрим $\triangle v_0v_1O$, где $O$ - середина многоугольника. Пусть точка попала в него. Тогда когда расстояние от неё до стороны $v_0v_1$ ближе, чем до диагоналей? Необходимое (но не достаточное) условие этому: точка должна лежать в треугольнике, образованном пересечением диагоналей $v_0v_2$ и $v_1v_{n-1}$. Иначе бы путь до стороны $v_0v_1$ состоял бы из пересечения с какой-либо из этих диагоналей, а значит расстояние до стороны $v_0v_1$ явно не ближе, чем до диагоналей. На рисунке пересечение эти двух диагоналей я обозначил буквой $A$. Давайте найдём угол $\angle v_1v_0v_2$, обозначенный на рисунке как $\alpha$. Рассмотрим для этого $\triangle v_0v_1v_2$. $\angle v_1 = \frac{(n-2)\pi}{n}$ по известной со школы формуле углов правильного многоугольника. Так как этот треугольник равнобедренный, оставшиеся два угла равны, значит $$\angle v_0 = \angle v_2 = \frac{\pi - \frac{(n-2)\pi}{n}}{2} = \frac{\pi}{n} = \alpha$$. Знаем, что вершины, равноудаленные от $v_0v_1$ и $v_0A$, лежат на биссектрисе угла, образованного этими сторонами. Обозначим пересечение биссектрис из $\angle Av_1v_0$ и $\angle v_1v_0A$ как $B$. Получаем, что все вершины, лежащие в $\triangle v_0v_1B$ находятся ближе к стороне $v_0v_1$, чем к диагоналям многоугольника. Осталось найти его площадь. Опустим из $B$ высоту на сторону многоугольник, обозначив точку пересечения как $v_{0.5}$. Обозначим длину этой высоты как $h$. Тогда: 
$$\frac{h}{(\frac{s}{2})}=\tg \frac{\pi}{2n}$$
$$2h=s\tg \frac{\pi}{2n}$$
$$h=\frac{s\tg \frac{\pi}{2n}}{2}$$
$$S_{\triangle v_0v_{0.5}B}=\frac{\frac{s\tg \frac{\pi}{2n}}{2} \cdot \frac{s}{2}}{2}$$
$$S_{\triangle v_0v_1B}=\frac{s\tg \frac{\pi}{2n}}{2} \cdot \frac{s}{2}=\frac{s^2}{4} \tg \frac{\pi}{2n}$$
Площадь правильного многоугольника обозначим за $$S_n = \frac{n}{4} s^2 \ctg \frac{\pi}{n}$$
Тогда получаем $$P_n = \frac{nS_{\triangle v_0v_1B}}{S_n}= \frac{n\frac{s^2}{4} \tg \frac{\pi}{2n}}{\frac{n}{4} s^2 \ctg \frac{\pi}{n}}=\tg \frac{\pi}{2n} \tg \frac{\pi}{n}=\frac{1-\cos{\frac{\pi}{n}}}{\cos \frac{\pi}{n}} = \frac{1}{\cos{\frac{\pi}{n}}}- 1$$
Теперь перейдём к асимптотике. Так как у нас по условию $n \to \infty$, то $\frac{\pi}{n} \to 0$. Тогда
$$P_n =\tg \frac{\pi}{2n} \tg \frac{\pi}{n} \sim \frac{\pi}{2n} \cdot \frac{\pi}{n} = \frac{\pi^2}{2}n^{-2}$$
$$P_n = \frac{\pi^2}{2}n^{-2}(1+o(1))$$
Значит 
$$C=\frac{\pi^2}{2}, a=-2$$
Перейдём к симуляции. Чтобы выбрать равномерно точку в правильном $n$-угольнике, достаточно выбирать точку в квадрате, куда вписан наш $n$-угольник, а если точка в него не попала, заново выбирать. Так делать, пока точка не попадёт в $n$-угольник. Действительно, никакая область этого $n$-угольника не получает преимуществ над другими из-за такого алгоритма, а значит мы равномерно выбираем точку внутри $n$-угольника.

In [23]:
import random
import math

random.seed("Sitnikov Sergey")


def create_poly(n):
    vertices = [
        (math.cos(2 * math.pi * k / n), math.sin(2 * math.pi * k / n)) for k in range(n)
    ]
    sides = [(vertices[i], vertices[(i + 1) % n]) for i in range(n)]
    diagonals = [
        (vertices[i], vertices[j])
        for i in range(n)
        for j in range(i + 1, n)
        if j - i != 1 and (i, j) != (0, n - 1)
    ]
    return sides, diagonals


def check_inside(point, sides):
    x, y = point
    for (ax, ay), (bx, by) in sides:
        cross = (bx - ax) * (y - ay) - (by - ay) * (x - ax)
        if cross < 0:
            return False
    return True


def from_point_to_segment(A, C, D):
    CD = (D[0] - C[0], D[1] - C[1])
    C__A = (A[0] - C[0], A[1] - C[1])
    DC = (C[0] - D[0], C[1] - D[1])
    A__D = (A[0] - D[0], A[1] - D[1])
    right_poluploskost = CD[0] * C__A[0] + CD[1] * C__A[1] >= 0
    left_poluploskost = DC[0] * A__D[0] + DC[1] * A__D[1] >= 0
    if left_poluploskost and right_poluploskost:
        cross = CD[0] * C__A[1] - CD[1] * C__A[0]
        length = math.sqrt(CD[0] ** 2 + CD[1] ** 2)
        return abs(cross) / length
    return min(
        math.sqrt(C__A[0] ** 2 + C__A[1] ** 2), math.sqrt(A__D[0] ** 2 + A__D[1] ** 2)
    )


def simulate_one(sides, diagonals):
    while True:
        point = (random.uniform(-1, 1), random.uniform(-1, 1))
        if check_inside(point, sides):
            break
    distance_boundary = min(
        from_point_to_segment(point, start, end) for start, end in sides
    )
    distance_diagonal = min(
        from_point_to_segment(point, start, end) for start, end in diagonals
    )
    return distance_boundary < distance_diagonal


COUNT_EXPERIMENTS = 200_000

print("n        simulation    formula       delta to formula")

for n in range(4, 15):
    sides, diagonals = create_poly(n)
    good_count = 0
    for _ in range(COUNT_EXPERIMENTS):
        good_count += simulate_one(sides, diagonals)
    simulation = good_count / COUNT_EXPERIMENTS
    formula = math.tan(math.pi / n) * math.tan(math.pi / (2 * n))
    delta = (simulation - formula) / formula * 100
    print(f"{n:<3} {simulation:>13.6f} " f"{formula:>13.6f} {delta:+10.2f}%")

n        simulation    formula       delta to formula
4        0.413125      0.414214      -0.26%
5        0.237110      0.236068      +0.44%
6        0.154610      0.154701      -0.06%
7        0.109020      0.109916      -0.82%
8        0.082655      0.082392      +0.32%
9        0.063665      0.064178      -0.80%
10       0.050145      0.051462      -2.56%
11       0.043130      0.042217      +2.16%
12       0.035820      0.035276      +1.54%
13       0.030010      0.029928      +0.27%
14       0.025380      0.025717      -1.31%


## Задача 3 (Вариант 1)
Пусть имеются две независимые серии испытаний Бернулли на $n$ опытов
в каждой с вероятностью успеха $p$, $S_i$ — количество успехов
в $n$ испытаниях в $i$-й серии. Найти вероятность
$$P(S_1 = k \mid S_1 + S_2 = m).$$
По формуле условной вероятности:
$$P(S_1 = k \mid S_1 + S_2 = m) = \frac{P(S_1 = k \; \& \; S_1 + S_2 = m)}{P(S_1 + S_2 = m)}$$
$$P(S_1 = k \; \& \; S_1 + S_2 = m) = P(S_1 = k \; \& \; S_2 = m - k)$$
Т.к. $S_1,S_2$ независимы, то $$P(S_1 = k \; \& \; S_1 + S_2 = m) = P(S_1 = k \; \& \; S_2 = m - k) = P(S_1=k)P(S_2=m-k)$$
Две независимве серии испытаний Бернулли на $n$ опытов - это всё равно, что одна серия испытания на $2n$ опытов.

$$P(S_1 = k \mid S_1 + S_2 = m) = \frac{P(S_1 = k \; \& \; S_1 + S_2 = m)}{P(S_1 + S_2 = m)}=\frac{P(S_1=k)P(S_2=m-k)}{P(S_1 + S_2 = m)}$$
$$=\frac{C_n^kp^kq^{n-k}C_n^{m-k}p^{m-k}q^{n-m+k}}{C_{2n}^mp^mq^{2n-m}}=\frac{C_n^kC_n^{m-k}}{C_{2n}^m}$$
С условиями: $$0 \le k \le n, \; 0 \le m-k \le n \Rightarrow \max(0,m-n) \le k \le \min(n,m)$$ 

## Задача 4 (Вариант 3)
Биномиальное распределение и нормальное (интервалы)

Из замечания к интегральной предельной теореме Муавра-Лапласа имеем, что для $-\infty \le A < B \le +\infty$ $$P(A < S_n \le B) - \left( \Phi\left(\frac{B-np}{\sqrt{npq}}\right) - \Phi\left(\frac{A-np}{\sqrt{npq}}\right) \right) \longrightarrow 0, \qquad n \to \infty.$$ 

In [24]:
import random
from statistics import NormalDist

random.seed("Sitnikov Sergey")


def simulate_one(n, p):
    successes = 0
    for _ in range(n):
        if random.random() < p:
            successes += 1
    return successes


def get(A, B, n, p):
    return NormalDist().cdf((B - n * p) / (n * p * (1 - p)) ** 0.5) - NormalDist().cdf(
        (A - n * p) / (n * p * (1 - p)) ** 0.5
    )


COUNT_EXPERIMENTS = 20_000

TESTS = {
    0.1: {
        100: [(7, 13), (4, 16), (10, 16)],
        300: [(25, 35), (20, 40), (30, 40)],
        500: [(44, 56), (37, 63), (50, 63)],
        1000: [(91, 109), (82, 118), (100, 118)],
        3000: [(284, 316), (268, 332), (300, 332)],
        10000: [(970, 1030), (940, 1060), (1000, 1060)],
    },
    0.5: {
        100: [(45, 55), (40, 60), (50, 60)],
        300: [(142, 158), (133, 167), (150, 167)],
        500: [(239, 261), (228, 272), (250, 272)],
        1000: [(485, 515), (469, 531), (500, 531)],
        3000: [(1473, 1527), (1446, 1554), (1500, 1554)],
        10000: [(4950, 5050), (4900, 5100), (5000, 5100)],
    },
    0.9: {
        100: [(87, 93), (84, 96), (90, 96)],
        300: [(265, 275), (260, 280), (270, 280)],
        500: [(444, 456), (437, 463), (450, 463)],
        1000: [(891, 909), (882, 918), (900, 918)],
        3000: [(2684, 2716), (2668, 2732), (2700, 2732)],
        10000: [(8970, 9030), (8940, 9060), (9000, 9060)],
    },
}

print("n     p   L    R         simulation    approx        delta")

for p in TESTS.keys():
    for n in TESTS[p].keys():
        sample = [simulate_one(n, p) for _ in range(COUNT_EXPERIMENTS)]
        for L, R in TESTS[p][n]:
            good_count = sum(L <= exp <= R for exp in sample)
            simulation = good_count / COUNT_EXPERIMENTS
            approx = get(L - 0.00001, R, n, p)
            delta = (simulation - approx) / simulation * 100
            print(
                f"{n:<5} {p:<3} {L:<4} {R:<4} {simulation:>13.6f} "
                f"{approx:>13.6f} {delta:+10.2f}%"
            )

n     p   L    R         simulation    approx        delta
100   0.1 7    13        0.757750      0.682690      +9.91%
100   0.1 4    16        0.971450      0.954500      +1.74%
100   0.1 10   16        0.526750      0.477251      +9.40%
300   0.1 25   35        0.714650      0.664077      +7.08%
300   0.1 20   40        0.957050      0.945708      +1.19%
300   0.1 30   40        0.497100      0.472855      +4.88%
500   0.1 44   56        0.669250      0.628907      +6.03%
500   0.1 37   63        0.954800      0.947368      +0.78%
500   0.1 50   63        0.496550      0.473684      +4.60%
1000  0.1 91   109       0.679100      0.657219      +3.22%
1000  0.1 82   118       0.945550      0.942220      +0.35%
1000  0.1 100  118       0.483650      0.471111      +2.59%
3000  0.1 284  316       0.687150      0.669809      +2.52%
3000  0.1 268  332       0.953700      0.948520      +0.54%
3000  0.1 300  332       0.480100      0.474260      +1.22%
10000 0.1 970  1030      0.690750      0.

## Задача 5 (Вариант 1)
Дана функция распределения:

![F](img/fifth.png)

2. Функция $F$ имеет скачки. В этих точках $p(x) \ne 0$, значит эти точки входят в дискретную часть. Такие скачки при $x_k \in \{-3, -2, -1, -0.5, 0.2, 0.4, 0.6, 1.5, 2, 3\}$ с вероятностями соответственно $p_k \in \{0.02, 0.03, 0.025, 0.045, 0.155, 0.055, 0.04, 0.05, 0.045, 0.055\}$. Имеем:
$$\alpha = \sum_{k=1}^{10} p_k = 0.52$$
$$1-\alpha = 0.48$$
$$F(x) = 0.52F_{disc}(x) + 0.48F_{cont}(x)$$

In [25]:
import random
import math

random.seed("Sitnikov Sergey")


def F_disc_raw(x):
    if x < -3:
        return 0
    elif x < -2:
        return 0.02
    elif x < -1:
        return 0.05
    elif x < -0.5:
        return 0.075
    elif x < 0.2:
        return 0.12
    elif x < 0.4:
        return 0.275
    elif x < 0.6:
        return 0.33
    elif x < 1.5:
        return 0.37
    elif x < 2:
        return 0.42
    elif x < 3:
        return 0.465
    else:
        return 0.52


def F_disc(x):
    return 1 / 0.52 * F_disc_raw(x)


def F_cont(x):
    if x < 0.2:
        return 0
    elif x < 1:
        return (x - 0.2) / 0.8
    else:
        return 1


def F(x):
    return 0.52 * F_disc(x) + 0.48 * F_cont(x)


def invF(t):
    if t == 0:
        return -math.inf
    elif 0 < t <= 0.02:
        return -3
    elif 0.02 < t <= 0.05:
        return -2
    elif 0.05 < t <= 0.075:
        return -1
    elif 0.075 < t <= 0.12:
        return -0.5
    elif 0.12 < t <= 0.275:
        return 0.2
    elif 0.275 < t <= 0.395:
        return (t - 0.155) / 0.6
    elif 0.395 < t <= 0.45:
        return 0.4
    elif 0.45 < t <= 0.57:
        return (t - 0.21) / 0.6
    elif 0.57 < t <= 0.61:
        return 0.6
    elif 0.61 < t <= 0.85:
        return (t - 0.25) / 0.6
    elif 0.85 < t <= 0.9:
        return 1.5
    elif 0.9 < t <= 0.945:
        return 2
    else:
        return 3


def build_walker(p):
    n = len(p)
    q = [n * pr for pr in p]
    small = [i for i in range(n) if q[i] < 1]
    large = [i for i in range(n) if q[i] >= 1]
    probs = [1.0] * n
    alias = list(range(n))
    while small and large:
        cur_small = small.pop()
        cur_large = large.pop()
        probs[cur_small] = q[cur_small]
        alias[cur_small] = cur_large
        q[cur_large] -= 1 - q[cur_small]
        if q[cur_large] < 1:
            small.append(cur_large)
        else:
            large.append(cur_large)
    return probs, alias


p = [
    i / 0.52 for i in [0.02, 0.03, 0.025, 0.045, 0.155, 0.055, 0.04, 0.05, 0.045, 0.055]
]

probs, alias = build_walker(p)


def get_walker(probs=probs, alias=alias):
    i = random.randrange(len(probs))
    u01 = random.random()
    if u01 < probs[i]:
        return i
    return alias[i]


def invFcont(x):
    return 0.2 + 0.8 * x


values = [-3, -2, -1, -0.5, 0.2, 0.4, 0.6, 1.5, 2, 3]


def get_quantile():
    return invF(1 - random.random())


def get_fdisc_fcont_decomp():
    if random.random() < 0.52:
        return values[get_walker()]
    return invFcont(random.random())


def F_simulate(sample, t):
    return sum(x <= t for x in sample) / len(sample)

In [26]:
COUNT_EXPERIMENTS = 200_000

TESTS = [
    -3.1,
    -3,
    -2.5,
    -2,
    -1.5,
    -1,
    -0.75,
    -0.5,
    0,
    0.199,
    0.2,
    0.3,
    0.399,
    0.4,
    0.5,
    0.599,
    0.6,
    0.8,
    1,
    1.4,
    1.5,
    1.9,
    2,
    2.5,
    3,
    3.1,
]

for name, simulate_one in [
    ("Quantile", get_quantile),
    ("Fdisc Fcont decomp", get_fdisc_fcont_decomp),
]:
    sample = [simulate_one() for _ in range(COUNT_EXPERIMENTS)]
    max_delta = 0
    print(name)
    print("t           simulation    formula       delta")
    for t in TESTS:
        simulation = F_simulate(sample, t)
        formula = F(t)
        delta = simulation - formula
        max_delta = max(max_delta, abs(delta))
        print(f"{t:<7.3f} {simulation:>13.6f} " f"{formula:>13.6f} {delta:+12.6f}")
    print(f"Max absolute delta: {max_delta:.6f}\n")

Quantile
t           simulation    formula       delta
-3.100       0.000000      0.000000    +0.000000
-3.000       0.019635      0.020000    -0.000365
-2.500       0.019635      0.020000    -0.000365
-2.000       0.049565      0.050000    -0.000435
-1.500       0.049565      0.050000    -0.000435
-1.000       0.074945      0.075000    -0.000055
-0.750       0.074945      0.075000    -0.000055
-0.500       0.120150      0.120000    +0.000150
0.000        0.120150      0.120000    +0.000150
0.199        0.120150      0.120000    +0.000150
0.200        0.275085      0.275000    +0.000085
0.300        0.334170      0.335000    -0.000830
0.399        0.393975      0.394400    -0.000425
0.400        0.449395      0.450000    -0.000605
0.500        0.509485      0.510000    -0.000515
0.599        0.569610      0.569400    +0.000210
0.600        0.611120      0.610000    +0.001120
0.800        0.730510      0.730000    +0.000510
1.000        0.849485      0.850000    -0.000515
1.400        0

5. Сравнение эффективности.
Обозначим размер выборки через $n$, число участков исходной функции распределения через $k$, а кол-во дискретных значений через $m$.
В алгоритме через квантильную функцию условия проверяются последовательно значит в худшем придется использовать для каждого значения $O(k)$ действий, для всей выборки - $O(nk)$ без предподсчета
В алгоритме с разбиением на $F_{disc}$ и $F_{cont}$ таблица Уолкера строится один раз за $O(m)$ времени и занимает $O(m)$ дополнительной памяти. Каждый индекс при подготовке добавляется в один из списков, а на каждом шаге один столбец окончательно заполняется. После подготовки генерация одного значения требует $O(1)$ действий. Для всей выборки получаем $O(m+n)$ времени.
Чтобы сравнить фактическую скорость, можно измерить время генерации для нескольких размеров выборки и усреднить по 5 запускам. Вариант с декомпозицией вызывает рандом два или три раза, в зависимости от того, генерируем мы дискретную величину или непрерывную, поэтому он не факт, что будет, вообще говоря, строго быстрее.


In [27]:
from time import perf_counter

COUNT_REPEATS = 5
COUNT_PREPARATIONS = 5_000

for _ in range(COUNT_PREPARATIONS):
    build_walker(p)

print("n         algorithm           time, sec             values per second")

times = {}
for n in [20_000, 200_000, 1_000_000]:
    times[n] = {}
    for name, simulate_one in [
        ("Quantile", get_quantile),
        ("Decomp", get_fdisc_fcont_decomp),
    ]:
        total_time = 0
        for _ in range(COUNT_REPEATS):
            random.seed("Sitnikov Sergey")
            start = perf_counter()
            for _ in range(n):
                simulate_one()
            total_time += perf_counter() - start
        average_time = total_time / COUNT_REPEATS
        times[n][name] = average_time
        speed = n / average_time
        print(f"{n:<9} {name:<12} {average_time:>15.6f} " f"{speed:>20.0f}")

ratio = times[1_000_000]["Decomp"] / times[1_000_000]["Quantile"]
if ratio >= 1:
    print(f"\nQuantile is faster by {ratio:.2f} times for n = 1_000_000.")
else:
    print(f"\nFdisc Fcont decomp is faster by {1 / ratio:.2f} times for n = 1_000_000.")

n         algorithm           time, sec             values per second
20000     Quantile            0.007905              2529927
20000     Decomp              0.005594              3575563
200000    Quantile            0.077221              2589981
200000    Decomp              0.052819              3786480
1000000   Quantile            0.381141              2623703
1000000   Decomp              0.273138              3661154

Fdisc Fcont decomp is faster by 1.40 times for n = 1_000_000.
